#**Task 1: Card Probability Problems**

In [1]:
import random


suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']
deck = [(rank, suit) for suit in suits for rank in ranks]
red_cards = [card for card in deck if card[1] in ['Hearts', 'Diamonds']]
print(f"P(Red Card): {len(red_cards)/len(deck)}")

hearts_in_red = [card for card in red_cards if card[1] == 'Hearts']
print(f"P(Heart | Red): {len(hearts_in_red)/len(red_cards)}")

face_cards = [card for card in deck if card[0] in ['Jack', 'Queen', 'King']]
diamonds_in_face = [card for card in face_cards if card[1] == 'Diamonds']
print(f"P(Diamond | Face): {len(diamonds_in_face)/len(face_cards)}")
spade_or_queen_in_face = [card for card in face_cards if card[1] == 'Spades' or card[0] == 'Queen']
print(f"P(Spade or Queen | Face): {len(spade_or_queen_in_face)/len(face_cards)}")

P(Red Card): 0.5
P(Heart | Red): 0.5
P(Diamond | Face): 0.25
P(Spade or Queen | Face): 0.5


In [6]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 12.6 MB/s eta 0:00:00


#**Task 2: Student Performance Bayesian Network**

In [19]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([('I', 'G'), ('S', 'G'), ('D', 'G'), ('G', 'P')])
cpd_i = TabularCPD('I', 2, [[0.7], [0.3]]) # High, Low
cpd_s = TabularCPD('S', 2, [[0.6], [0.4]]) # Sufficient, Insufficient
cpd_d = TabularCPD('D', 2, [[0.4], [0.6]]) # Hard, Easy

# P(G | I, S, D) - 2x2x2=8 combinations for parents
cpd_g = TabularCPD('G', 3, [
    [0.9, 0.7, 0.8, 0.4, 0.7, 0.3, 0.5, 0.1], # Grade A
    [0.08, 0.2, 0.15, 0.4, 0.2, 0.4, 0.3, 0.4], # Grade B
    [0.02, 0.1, 0.05, 0.2, 0.1, 0.3, 0.2, 0.5]  # Grade C
], evidence=['I', 'S', 'D'], evidence_card=[2, 2, 2])

# P(P | G)
cpd_p = TabularCPD('P', 2, [
    [0.95, 0.80, 0.50], # Pass = Yes
    [0.05, 0.20, 0.50]  # Pass = No
], evidence=['G'], evidence_card=[3])

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
infer = VariableElimination(model)
print("P(Pass | Study=Sufficient, Difficulty=Hard):")
print(infer.query(['P'], evidence={'S': 0, 'D': 0}))

print("\nP(Intelligence | Pass=Yes):")
print(infer.query(['I'], evidence={'P': 0}))

P(Pass | Study=Sufficient, Difficulty=Hard):
+------+----------+
| P    |   phi(P) |
+======+==========+
| P(0) |   0.9128 |
+------+----------+
| P(1) |   0.0872 |
+------+----------+

P(Intelligence | Pass=Yes):
+------+----------+
| I    |   phi(I) |
+======+==========+
| I(0) |   0.7256 |
+------+----------+
| I(1) |   0.2744 |
+------+----------+


#**Task 3: Disease Prediction Bayesian Network**

In [20]:
disease_model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'), ('Disease', 'Cough'),
    ('Disease', 'Fatigue'), ('Disease', 'Chills')
])
cpd_dis = TabularCPD('Disease', 2, [[0.3], [0.7]])
cpd_fever = TabularCPD('Fever', 2, [[0.9, 0.5], [0.1, 0.5]], evidence=['Disease'], evidence_card=[2])
cpd_cough = TabularCPD('Cough', 2, [[0.8, 0.6], [0.2, 0.4]], evidence=['Disease'], evidence_card=[2])
cpd_fatigue = TabularCPD('Fatigue', 2, [[0.7, 0.3], [0.3, 0.7]], evidence=['Disease'], evidence_card=[2])
cpd_chills = TabularCPD('Chills', 2, [[0.6, 0.4], [0.4, 0.6]], evidence=['Disease'], evidence_card=[2])

disease_model.add_cpds(cpd_dis, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)
d_infer = VariableElimination(disease_model)

print(d_infer.query(['Disease'], evidence={'Fever': 0, 'Cough': 0}))

print(d_infer.query(['Disease'], evidence={'Fever': 0, 'Cough': 0, 'Chills': 0}))

print(d_infer.query(['Fatigue'], evidence={'Disease': 0}))

+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.5070 |
+------------+----------------+
| Disease(1) |         0.4930 |
+------------+----------------+
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.6067 |
+------------+----------------+
| Disease(1) |         0.3933 |
+------------+----------------+
+------------+----------------+
| Fatigue    |   phi(Fatigue) |
+============+================+
| Fatigue(0) |         0.7000 |
+------------+----------------+
| Fatigue(1) |         0.3000 |
+------------+----------------+


#**Task 4: Weather Markov Model**

In [22]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
transition_matrix = np.array([
    [0.7, 0.2, 0.1],  # From Sunny
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5]   # From Rainy
])

def simulate_weather(num_days=10, start_state=0):
    current_state = start_state
    sequence = [states[current_state]]

    rainy_count = 1 if states[current_state] == "Rainy" else 0

    for _ in range(num_days - 1):
        current_state = np.random.choice(
            [0, 1, 2], p=transition_matrix[current_state]
        )
        sequence.append(states[current_state])
        if states[current_state] == "Rainy":
            rainy_count += 1

    return sequence, rainy_count

np.random.seed(42)
sequence, count = simulate_weather(10)
print("Sample 10-day forecast:", " → ".join(sequence))
print(f"Rainy days in sample: {count}\n")

trials = 10000
at_least_3_rainy = sum(
    1 for _ in range(trials) if simulate_weather(10)[1] >= 3
)

print(f"Trials run         : {trials}")
print(f"P(≥3 rainy days)   : {at_least_3_rainy / trials:.4f}")

Sample 10-day forecast: Sunny → Sunny → Rainy → Rainy → Rainy → Sunny → Sunny → Sunny → Cloudy → Cloudy
Rainy days in sample: 3

Trials run         : 10000
P(≥3 rainy days)   : 0.3472
